POORVIKA G K

NOVEMBER BATCH

25/11/2026

In [2]:
import numpy as np
from datetime import datetime, timedelta
FILE_PATH = 'hostel_bois.txt'

In [3]:
def is_message_line(line):
    """True if the line starts with the date pattern NN/NN/NN, (first 8 chars) -> a new message."""
    return (len(line) > 9 and line[2] == '/' and line[5] == '/' and line[8] == ','
            and line[0:2].isdigit() and line[3:5].isdigit() and line[6:8].isdigit())


def classify_text(text):
    """Return 'media', 'deleted' or 'text' for a message body."""
    t = text.strip()
    if t == '<Media omitted>':
        return 'media'
    if t == 'This message was deleted' or t == 'You deleted this message':
        return 'deleted'
    return 'text'


def parse_chat(path):
    """
    Parse a WhatsApp .txt export.
    Returns (messages, info) where
      messages = list of dicts {'timestamp','sender','text','dt','kind'}  (system lines NOT included)
      info     = dict with the system counter, group name and file status.
    """
    info = {'system': 0, 'group_name': 'Unknown Group', 'ok': True, 'bad_lines': 0}
    messages = []
    try:
        with open(path, 'r', encoding='utf-8') as f:
            content = f.read()
    except FileNotFoundError:
        info['ok'] = False
        return messages, info

    lines = content.split('\n')
    last = None            # the last real message dict (for multi-line continuation)

    for raw in lines:
        line = raw.replace('\r', '').replace('\u200e', '')   # clean stray carriage returns / invisible marks
        if line.strip() == '':                                  # e) empty lines -> skip silently
            continue

        if not is_message_line(line):                           # d) multi-line: continuation of previous message
            if last is not None:
                last['text'] = last['text'] + ' ' + line.strip()
            continue

        parts = line.split(' - ', 1)                            # timestamp | rest
        if len(parts) < 2:
            info['bad_lines'] += 1
            last = None
            continue
        timestamp, rest = parts[0].strip(), parts[1]

        if ': ' not in rest:                                    # a) system message: no "Sender: " part
            info['system'] += 1
            last = None
            if 'created group "' in rest:                       # grab the group name from the system line
                start = rest.find('"') + 1
                end = rest.rfind('"')
                if end > start:
                    info['group_name'] = rest[start:end]
            continue

        sender, text = rest.split(': ', 1)                      # split only once (message may contain ': ')
        try:
            dt = datetime.strptime(timestamp, '%d/%m/%y, %H:%M')
        except ValueError:
            info['bad_lines'] += 1
            last = None
            continue

        msg = {'timestamp': timestamp, 'sender': sender.strip(), 'text': text.strip(),
               'dt': dt, 'kind': classify_text(text)}
        messages.append(msg)
        last = msg

    # kind may change if a continuation was appended (it never changes media/deleted, but re-check for safety)
    for m in messages:
        m['kind'] = classify_text(m['text'])
    return messages, info


messages, info = parse_chat(FILE_PATH)

# If the file is missing and we are on Colab, ask for an upload
if not info['ok']:
    try:
        from google.colab import files
        print('hostel_bois.txt not found - please upload it now...')
        uploaded = files.upload()
        FILE_PATH = list(uploaded.keys())[0]
        messages, info = parse_chat(FILE_PATH)
    except ImportError:
        print('File not found: place hostel_bois.txt next to this notebook.')

n_media   = sum(1 for m in messages if m['kind'] == 'media')
n_deleted = sum(1 for m in messages if m['kind'] == 'deleted')
n_text    = sum(1 for m in messages if m['kind'] == 'text')
participants = sorted(set(m['sender'] for m in messages))
all_dates = sorted(set(m['dt'].date() for m in messages))
n_days = (all_dates[-1] - all_dates[0]).days + 1 if all_dates else 0

print(f"Successfully parsed {len(messages):,} messages from {len(participants)} participants over {n_days} days, "
      f"skipped {info['system']} system messages, {n_media} media-omitted, {n_deleted} deleted messages "
      f"(these last two are counted per person but excluded from word / length stats).")

Successfully parsed 3,174 messages from 6 participants over 60 days, skipped 4 system messages, 32 media-omitted, 15 deleted messages (these last two are counted per person but excluded from word / length stats).


In [4]:
print("FIRST 5 MESSAGES")
for m in messages[:5]:
    print(f"  {m['timestamp']} | {m['sender']:<6} | {m['text'][:60]}")
print("\nLAST 5 MESSAGES")
for m in messages[-5:]:
    print(f"  {m['timestamp']} | {m['sender']:<6} | {m['text'][:60]}")

FIRST 5 MESSAGES
  01/04/24, 01:17 | Rahul  | scene fix
  01/04/24, 01:17 | Rahul  | haan
  01/04/24, 01:18 | Rahul  | kya scene
  01/04/24, 02:13 | Rahul  | abhi free hai?
  01/04/24, 02:13 | Rahul  | abey

LAST 5 MESSAGES
  30/05/24, 19:14 | Priya  | Take care everyone
  30/05/24, 19:28 | Priya  | Karan that sounds tough, take care
  30/05/24, 21:17 | Aman   | the existential dread is back
  30/05/24, 21:30 | Karan  | Long day guys, woke up at six for that placement workshop wh
  30/05/24, 23:31 | Aman   | anyone awake?


In [6]:
PUNCT = '.,!?;:"\'()[]{}<>-_*~`@#$%^&+=/\\|\u2018\u2019\u201c\u201d'

def count_into(counter, key, amount=1):
    """Our own mini-Counter: add `amount` to counter[key]."""
    counter[key] = counter.get(key, 0) + amount

def tokenize(text):
    """lowercase -> split -> strip punctuation -> drop empty tokens."""
    words = []
    for w in text.lower().split():
        w = w.strip(PUNCT)
        if w != '':
            words.append(w)
    return words

def pad_bar(value, max_value, width=20, char='█'):
    """Horizontal bar of block characters, length proportional to value."""
    if max_value <= 0 or value <= 0:
        return '.'
    return char * max(1, round(width * value / max_value))

def fmt_date(d, pattern='%d %B %Y'):
    return d.strftime(pattern)

# ---- per-person stats -------------------------------------------------------
msg_count   = {}   # every message the person sent (text + media + deleted)
media_count = {}
del_count   = {}
word_total  = {}   # words in real text messages
text_count  = {}

for m in messages:
    p = m['sender']
    count_into(msg_count, p)
    if m['kind'] == 'media':
        count_into(media_count, p)
    elif m['kind'] == 'deleted':
        count_into(del_count, p)
    else:
        count_into(text_count, p)
        count_into(word_total, p, len(m['text'].split()))

# people ranked by messages (highest first) -> row order for heatmap etc.
ranked_people = sorted(msg_count, key=lambda p: msg_count[p], reverse=True)
total_msgs = len(messages)

# ---- extra Day-2 stats: word-count & avg message length per person ----------
print(f"{'Member':<8}{'Messages':>9}{'Words':>8}{'Avg words/msg':>15}{'Media':>7}{'Deleted':>9}")
for p in ranked_people:
    avg_w = word_total.get(p, 0) / text_count[p] if text_count.get(p, 0) else 0
    print(f"{p:<8}{msg_count[p]:>9}{word_total.get(p,0):>8}{avg_w:>15.1f}{media_count.get(p,0):>7}{del_count.get(p,0):>9}")

Member   Messages   Words  Avg words/msg  Media  Deleted
Rahul         953    2399            2.6      7        6
Priya         718    3560            5.0      4        2
Neha          635    3317            5.3      8        3
Aman          490    2430            5.0      4        2
Karan         354   19681           57.0      7        2
Vikas          24      40            1.8      2        0


In [7]:
def print_overview():
    first, last = messages[0]['dt'], messages[-1]['dt']
    print('=' * 60)
    print(' GROUP OVERVIEW')
    print('=' * 60)
    print(f" Group          : {info['group_name']}")
    print(f" Period         : {fmt_date(first)} to {fmt_date(last)} ({n_days} days)")
    print(f" Total messages : {total_msgs:,}")
    print(f" Participants   : {len(participants)}")
    print()
    print(' MESSAGES PER PERSON')
    for p in ranked_people:
        pct = 100 * msg_count[p] / total_msgs
        print(f"   {p:<8}: {msg_count[p]:>5} ({pct:>4.1f}%)")

print_overview()

 GROUP OVERVIEW
 Group          : Hostel Bois 4ever
 Period         : 01 April 2024 to 30 May 2024 (60 days)
 Total messages : 3,174
 Participants   : 6

 MESSAGES PER PERSON
   Rahul   :   953 (30.0%)
   Priya   :   718 (22.6%)
   Neha    :   635 (20.0%)
   Aman    :   490 (15.4%)
   Karan   :   354 (11.2%)
   Vikas   :    24 ( 0.8%)


In [8]:
def busiest_day_and_hour():
    per_day, per_hour = {}, {}
    for m in messages:
        count_into(per_day, m['dt'].date())
        count_into(per_hour, m['dt'].hour)
    best_day = max(per_day, key=lambda d: per_day[d])
    best_hour = max(per_hour, key=lambda h: per_hour[h])
    avg_per_day = per_hour[best_hour] / n_days
    return best_day, per_day[best_day], best_hour, per_hour[best_hour], avg_per_day, per_day, per_hour

b_day, b_day_n, b_hour, b_hour_n, b_hour_avg, per_day_counts, per_hour_counts = busiest_day_and_hour()
print(f" Busiest day  : {fmt_date(b_day)} ({b_day_n} messages)")
print(f" Busiest hour : {b_hour:02d}:00 - {(b_hour + 1) % 24:02d}:00 "
      f"({b_hour_n} messages in total, avg {b_hour_avg:.0f} per day)")

 Busiest day  : 04 May 2024 (76 messages)
 Busiest hour : 18:00 - 19:00 (248 messages in total, avg 4 per day)


In [9]:
person_index = {p: i for i, p in enumerate(ranked_people)}
heat = np.zeros((len(ranked_people), 24), dtype=int)
for m in messages:
    heat[person_index[m['sender']], m['dt'].hour] += 1

print("Matrix shape:", heat.shape)
print("Row totals match message counts:",
      all(int(heat[person_index[p]].sum()) == msg_count[p] for p in ranked_people))

NIGHT_HOURS = [23, 0, 1, 2, 3, 4]          # 23:00 - 04:59

def night_percent(person):
    row = heat[person_index[person]]
    total = row.sum()
    return float(100 * row[NIGHT_HOURS].sum() / total) if total else 0.0

def shade(value, row_max):
    """4 shading levels relative to the person's own max hour."""
    if row_max == 0:
        return '. '
    r = value / row_max
    if r < 0.25:
        return '. '
    elif r < 0.50:
        return '░ '
    elif r < 0.75:
        return '▒ '
    return '█ '

def print_heatmap(tags=None):
    tags = tags or {}
    print(' ACTIVITY HEATMAP (hour of day, columns 00 to 23)')
    print('          ' + ''.join(f"{h:02d}    " for h in range(0, 24, 3)))
    for p in ranked_people:
        row = heat[person_index[p]]
        cells = ''.join(shade(v, row.max()) for v in row)
        tag = f"  <- {tags[p]}" if p in tags else ''
        print(f"   {p:<7}{cells}{tag}")

print_heatmap()

Matrix shape: (6, 24)
Row totals match message counts: True
 ACTIVITY HEATMAP (hour of day, columns 00 to 23)
          00    03    06    09    12    15    18    21    
   Rahul  . . . . . . . . . . . . ▒ ░ ░ ▒ ▒ ░ █ ▒ ░ █ ▒ ▒ 
   Priya  . . . . . . . ░ ▒ █ █ █ █ ▒ ▒ ░ ░ ▒ ▒ █ ▒ ░ ░ . 
   Neha   . . . . . ░ . . ▒ █ █ ░ ▒ ▒ ░ . ▒ █ █ █ ▒ ░ ░ ░ 
   Aman   ▒ █ █ ▒ █ . . . . . . . . . . . . . . . . . . ▒ 
   Karan  . . . . . . . . ░ ░ ▒ ░ █ ▒ █ ▒ ▒ ▒ ▒ █ ▒ ░ . . 
   Vikas  . . . . . . . ░ █ ░ ░ . ▒ ▒ . ░ ░ █ ▒ ▒ ░ ░ ░ ▒ 


In [10]:
STOP_WORDS = set([
    'i', 'is', 'the', 'a', 'an', 'and', 'or', 'to', 'of', 'in', 'on', 'for', 'at', 'it', 'its', 'my', 'me',
    'you', 'your', 'we', 'he', 'she', 'they', 'this', 'that', 'was', 'are', 'be', 'been', 'am', 'so', 'if',
    'but', 'with', 'as', 'by', 'do', 'did', 'not', 'no', 'have', 'has', 'had', 'will', 'just', 'what',
    'from', 'there', 'then', 'than', 'too', 'can', 'all', 'about', 'up', 'out', 'when', 'who', 'how',
    'hai', 'ho', 'ka', 'ki', 'ke', 'ko', 'se', 'ne', 'hi', 'bhi', 'toh', 'to', 'na', 'ya', 'h', 'u', 's'
])

# Design decision: for the GROUP-wide list we ignore very long "story" messages (> 30 words) so that one
# storyteller's rambling cannot swamp the group's slang. Per-person lists below use every text message.
LONG_MSG_WORDS = 30

word_freq = {}                 # group-wide
person_word_freq = {}          # person -> {word: count}
for m in messages:
    if m['kind'] != 'text':
        continue
    is_long = len(m['text'].split()) > LONG_MSG_WORDS
    for w in tokenize(m['text']):
        if w in STOP_WORDS or len(w) < 2:
            continue
        if not is_long:
            count_into(word_freq, w)
        pf = person_word_freq.setdefault(m['sender'], {})
        count_into(pf, w)

top_words = sorted(word_freq.items(), key=lambda kv: kv[1], reverse=True)[:10]

def print_top_words():
    print(" THIS GROUP'S FAVOURITE WORDS")
    biggest = top_words[0][1]
    for w, c in top_words:
        print(f"   {w:<10}{pad_bar(c, biggest):<21}{c:>5}")

print_top_words()
print("\n TOP 5 WORDS PER PERSON")
for p in ranked_people:
    best = sorted(person_word_freq.get(p, {}).items(), key=lambda kv: kv[1], reverse=True)[:5]
    print(f"   {p:<7}: " + ', '.join(f"{w} ({c})" for w, c in best))

 THIS GROUP'S FAVOURITE WORDS
   guys      ████████████████████   198
   bhai      ████████████████       160
   everyone  ████████████████       157
   scene     ███████████████        145
   please    ██████████████         141
   anyone    ██████████████         139
   yaar      ██████████████         139
   kya       █████████████          133
   sleep     ███████████            112
   ja        ██████████             101

 TOP 5 WORDS PER PERSON
   Rahul  : bhai (159), scene (144), kya (133), yaar (105), ja (101)
   Priya  : please (141), everyone (137), aman (93), anyone (90), okay (80)
   Neha   : guys (101), cant (58), ok (52), today (48), way (48)
   Aman   : sleep (71), i'm (56), anyone (49), wonder (43), night (41)
   Karan  : his (217), which (202), telling (179), today (171), started (150)
   Vikas  : haha (4), sorry (3), busy (3), tha (3), haan (3)


In [11]:
def compute_response_times():
    """gap (seconds) from the last message of a different person to this person's reply."""
    gaps = {}
    for i in range(1, len(messages)):
        prev, cur = messages[i - 1], messages[i]
        if cur['sender'] != prev['sender']:
            secs = (cur['dt'] - prev['dt']).total_seconds()
            gaps.setdefault(cur['sender'], []).append(secs)
    return {p: sum(g) / len(g) for p, g in gaps.items() if len(g) > 0}

def human_time(secs):
    if secs < 3600:
        return f"{secs / 60:.1f} minutes"
    return f"{secs / 3600:.1f} hours"

def compute_silent_streaks():
    """For each person: (longest streak length, start date, end date) of days with zero messages."""
    active_days = {p: set() for p in participants}
    for m in messages:
        active_days[m['sender']].add(m['dt'].date())

    all_days = [all_dates[0] + timedelta(days=i) for i in range(n_days)]
    result = {}
    for p in participants:
        best_len, best_start, best_end = 0, None, None
        cur_len, cur_start = 0, None
        for d in all_days:
            if d in active_days[p]:
                cur_len, cur_start = 0, None
            else:
                if cur_len == 0:
                    cur_start = d
                cur_len += 1
                if cur_len > best_len:
                    best_len, best_start, best_end = cur_len, cur_start, d
        result[p] = (best_len, best_start, best_end)
    return result, active_days

avg_response = compute_response_times()
streaks, active_days = compute_silent_streaks()

def print_response_patterns():
    fastest = min(avg_response, key=lambda p: avg_response[p])
    slowest = max(avg_response, key=lambda p: avg_response[p])
    print(' RESPONSE PATTERNS')
    print(f"   Fastest replier : {fastest} (avg {human_time(avg_response[fastest])})")
    print(f"   Slowest replier : {slowest} (avg {human_time(avg_response[slowest])})")
    print()
    print(' LONGEST SILENT STREAKS (consecutive days with zero messages)')
    for p in sorted(streaks, key=lambda p: streaks[p][0], reverse=True):
        length, s, e = streaks[p]
        if length == 0:
            print(f"   {p:<8}: 0 days (never went silent)")
        elif length >= 4:
            print(f"   {p:<8}: {length} days ({fmt_date(s, '%d %b')} to {fmt_date(e, '%d %b')})")
        else:
            print(f"   {p:<8}: {length} day{'s' if length > 1 else ''}")

print_response_patterns()

 RESPONSE PATTERNS
   Fastest replier : Rahul (avg 34.9 minutes)
   Slowest replier : Aman (avg 55.4 minutes)

 LONGEST SILENT STREAKS (consecutive days with zero messages)
   Vikas   : 11 days (23 Apr to 03 May)
   Aman    : 0 days (never went silent)
   Karan   : 0 days (never went silent)
   Neha    : 0 days (never went silent)
   Priya   : 0 days (never went silent)
   Rahul   : 0 days (never went silent)


In [12]:
CARING_KEYWORDS = ['okay', 'safe', 'eat', 'sleep', 'take care', 'are you', 'please', 'reminder',
                   'drink water', "don't forget"]
LAUGH_WORDS = ['lol', 'lmao', 'haha', 'rofl', 'lmfao']

def msgs_of(person, kind=None):
    return [m for m in messages if m['sender'] == person and (kind is None or m['kind'] == kind)]

# ---- one detection function per archetype: returns the raw metric -----------
def metric_spammer(person):
    """Average burst length: person's messages / number of back-to-back runs they started."""
    runs, prev_sender = 0, None
    for m in messages:
        if m['sender'] == person and prev_sender != person:
            runs += 1
        prev_sender = m['sender']
    return msg_count.get(person, 0) / runs if runs else 0.0

def metric_group_mom(person):
    """Total number of caring-keyword hits in the person's text messages."""
    hits = 0
    for m in msgs_of(person, 'text'):
        t = m['text'].lower()
        for kw in CARING_KEYWORDS:
            hits += t.count(kw)
    return hits

def metric_night_owl(person):
    """% of messages between 23:00 and 04:59 (taken from the NumPy heatmap)."""
    return night_percent(person)

def metric_storyteller(person):
    """Average words per real text message."""
    return word_total.get(person, 0) / text_count[person] if text_count.get(person, 0) else 0.0

def metric_drama_queen(person):
    """% of text messages that are ALL CAPS (>=3 chars, letters only checked by isupper) OR contain 2+ '!'."""
    texts = msgs_of(person, 'text')
    if not texts:
        return 0.0
    hits = 0
    for m in texts:
        t = m['text']
        if (len(t) >= 3 and t.isupper()) or t.count('!') >= 2:
            hits += 1
    return 100 * hits / len(texts)

def metric_ghost(person):
    """% of days in the chat period on which the person sent nothing."""
    return 100 * (n_days - len(active_days.get(person, set()))) / n_days if n_days else 0.0

def metric_comedian(person):
    """% of the person's text messages containing a laugh word."""
    texts = msgs_of(person, 'text')
    if not texts:
        return 0.0
    hits = 0
    for m in texts:
        t = m['text'].lower()
        if any(lw in t for lw in LAUGH_WORDS):
            hits += 1
    return 100 * hits / len(texts)

def metric_question_master(person):
    """% of text messages that end with '?'."""
    texts = msgs_of(person, 'text')
    if not texts:
        return 0.0
    return 100 * sum(1 for m in texts if m['text'].endswith('?')) / len(texts)

# ---- 9th (bonus) archetype: invented for Indian college life -----------------
def metric_punctual(person):
    """
    THE PAKKA PUNCTUAL ONE: the 'sleeps on time' topper. Rarely texts at night (<5% of messages between
    23:00-04:59) AND shows up on at least 80% of the days. Returns 1 if both hold else 0.
    Because there is no reason to beat a stronger trait, it is scored as a fallback (0.99 < 1.0 below).
    """
    active_pct = 100 * len(active_days.get(person, set())) / n_days if n_days else 0
    return 1.0 if (night_percent(person) < 5 and active_pct >= 80) else 0.0

FALLBACK = ['THE COMEDIAN', 'THE QUESTION MASTER', 'THE PAKKA PUNCTUAL ONE']

# archetype name -> (function, threshold). Order = tie-break priority.
ARCHETYPES = [
    ('THE GROUP MOM',       metric_group_mom,       None),   # relative: highest keyword count
    ('THE SPAMMER',         metric_spammer,         3.0),
    ('THE NIGHT OWL',       metric_night_owl,       60.0),
    ('THE STORYTELLER',     metric_storyteller,     30.0),
    ('THE DRAMA QUEEN',     metric_drama_queen,     30.0),
    ('THE GHOST',           metric_ghost,           60.0),
    ('THE COMEDIAN',        metric_comedian,        None),   # relative: highest laugh %
    ('THE QUESTION MASTER', metric_question_master, 25.0),
    ('THE PAKKA PUNCTUAL ONE', metric_punctual,     None),   # bonus 9th archetype
]

raw = {a: {p: fn(p) for p in participants} for a, fn, _ in ARCHETYPES}

def score_of(archetype, person, threshold):
    value = raw[archetype][person]
    if archetype == 'THE PAKKA PUNCTUAL ONE':
        return 0.99 if value == 1.0 else 0.0          # fallback-only
    if threshold is None:                              # relative archetypes: must be the top person (and > 0)
        best = max(raw[archetype].values())
        return 1.0 if (best > 0 and value == best) else (value / best if best > 0 else 0.0) * 0.99
    return value / threshold

scores = {a: {p: score_of(a, p, th) for p in participants} for a, fn, th in ARCHETYPES}
priority = {a: i for i, (a, fn, th) in enumerate(ARCHETYPES)}

def assign_archetypes():
    assigned, used = {}, set()
    # PRIMARY tier = the six main archetypes. FALLBACK tier (Comedian, Question Master, Punctual One) is only
    # used for people who did not win a primary archetype (the brief calls them tiebreaker / fallback).
    pairs = [(scores[a][p], p, a) for a, _, _ in ARCHETYPES if a not in FALLBACK
             for p in participants if scores[a][p] >= 1.0]
    pairs.sort(key=lambda t: (-t[0], priority[t[2]], t[1]))
    for sc, p, a in pairs:                              # step 1-2: qualified pairs, highest score first
        if p not in assigned and a not in used:
            assigned[p] = a
            used.add(a)
    for p in participants:                              # step 3: fallback for anyone left
        if p not in assigned:
            options = sorted([a for a, _, _ in ARCHETYPES if a not in used],
                             key=lambda a: (a not in FALLBACK, -scores[a][p], priority[a]))
            assigned[p] = options[0]
            used.add(options[0])
    runner_up = {}                                      # step 4: bonus runner-up
    for p in participants:
        others = sorted([a for a, _, _ in ARCHETYPES if a != assigned[p]],
                        key=lambda a: (-scores[a][p], priority[a]))
        runner_up[p] = others[0]
    return assigned, runner_up

archetype_of, runner_up_of = assign_archetypes()

def evidence(person, archetype):
    r = raw[archetype][person]
    if archetype == 'THE SPAMMER':           return f"avg {r:.1f} msgs in a row"
    if archetype == 'THE GROUP MOM':         return f"caring keyword score: {int(r)}"
    if archetype == 'THE NIGHT OWL':         return f"{r:.1f}% msgs between 23h-04h"
    if archetype == 'THE STORYTELLER':       return f"avg {r:.1f} words per msg"
    if archetype == 'THE DRAMA QUEEN':       return f"{r:.1f}% ALL-CAPS / !! messages"
    if archetype == 'THE GHOST':             return f"silent on {n_days - len(active_days[person])} of {n_days} days"
    if archetype == 'THE COMEDIAN':          return f"{r:.1f}% msgs with laugh words"
    if archetype == 'THE QUESTION MASTER':   return f"{r:.1f}% msgs end with '?'"
    return f"only {night_percent(person):.1f}% late-night msgs, shows up most days"

print(' PERSONALITY ARCHETYPES')
for p in ranked_people:
    a = archetype_of[p]
    print(f"   {p:<7}→ {a:<14} ({evidence(p, a)})   [runner-up: {runner_up_of[p]}]")

 PERSONALITY ARCHETYPES
   Rahul  → THE SPAMMER    (avg 4.5 msgs in a row)   [runner-up: THE NIGHT OWL]
   Priya  → THE GROUP MOM  (caring keyword score: 621)   [runner-up: THE QUESTION MASTER]
   Neha   → THE DRAMA QUEEN (63.3% ALL-CAPS / !! messages)   [runner-up: THE PAKKA PUNCTUAL ONE]
   Aman   → THE NIGHT OWL  (79.8% msgs between 23h-04h)   [runner-up: THE SPAMMER]
   Karan  → THE STORYTELLER (avg 57.0 words per msg)   [runner-up: THE PAKKA PUNCTUAL ONE]
   Vikas  → THE GHOST      (silent on 44 of 60 days)   [runner-up: THE COMEDIAN]


In [13]:
def final_report():
    W = 60
    first, last = messages[0]['dt'], messages[-1]['dt']
    owl_tags = {p: archetype_of[p] for p in ranked_people}

    print('=' * W)
    print(f" 🧬 GROUPDNA REPORT — \"{info['group_name']}\"")
    print(f" {n_days} days • {total_msgs:,} messages • {len(participants)} members")
    print('=' * W)
    print(f" Period      : {fmt_date(first)} to {fmt_date(last)}")
    print(f" Busiest day : {fmt_date(b_day)} ({b_day_n} messages)")
    print(f" Busiest hour: {b_hour:02d}:00 - {(b_hour + 1) % 24:02d}:00")
    print(f" Media shared: {n_media}   Deleted: {n_deleted}   System msgs: {info['system']}")
    print('-' * W)

    print(' MESSAGES PER PERSON')
    top = msg_count[ranked_people[0]]
    for p in ranked_people:
        pct = 100 * msg_count[p] / total_msgs
        print(f"   {p:<7}{pad_bar(msg_count[p], top):<21}{msg_count[p]:>5} ({pct:>4.1f}%)")
    print('-' * W)

    print_heatmap(owl_tags)
    print('-' * W)

    print_top_words()
    print('-' * W)

    print_response_patterns()
    print('-' * W)

    print(' PERSONALITY ARCHETYPES')
    for p in ranked_people:
        a = archetype_of[p]
        print(f"   {p:<7}→ {a} ({evidence(p, a)})")
    print('=' * W)
    print(' Generated by GroupDNA • Built with Python + NumPy')
    print('=' * W)

final_report()

 🧬 GROUPDNA REPORT — "Hostel Bois 4ever"
 60 days • 3,174 messages • 6 members
 Period      : 01 April 2024 to 30 May 2024
 Busiest day : 04 May 2024 (76 messages)
 Busiest hour: 18:00 - 19:00
 Media shared: 32   Deleted: 15   System msgs: 4
------------------------------------------------------------
 MESSAGES PER PERSON
   Rahul  ████████████████████   953 (30.0%)
   Priya  ███████████████        718 (22.6%)
   Neha   █████████████          635 (20.0%)
   Aman   ██████████             490 (15.4%)
   Karan  ███████                354 (11.2%)
   Vikas  █                       24 ( 0.8%)
------------------------------------------------------------
 ACTIVITY HEATMAP (hour of day, columns 00 to 23)
          00    03    06    09    12    15    18    21    
   Rahul  . . . . . . . . . . . . ▒ ░ ░ ▒ ▒ ░ █ ▒ ░ █ ▒ ▒   <- THE SPAMMER
   Priya  . . . . . . . ░ ▒ █ █ █ █ ▒ ▒ ░ ░ ▒ ▒ █ ▒ ░ ░ .   <- THE GROUP MOM
   Neha   . . . . . ░ . . ▒ █ █ ░ ▒ ▒ ░ . ▒ █ █ █ ▒ ░ ░ ░   <- THE DRAMA QUEEN
   Am